# Assignment 2: Retrieval-Augmented Generation Question Answering

Welcome to the second assignment for 50.055 Machine Learning Operations This is a continuation of your work in assignment 1, where you will be working to build a chatbot which can answer questions about SUTD to prospective students.


**This assignment is a individual assignment.**

- Read the instructions in this notebook carefully
- Add your solution code and answers in the appropriate places. The questions are marked as **QUESTION:**, the places where you need to add your code and text answers are marked as **ADD YOUR SOLUTION HERE**
- The completed notebook, including your added code and generated output will be your submission for the assignment.
- The notebook should execute without errors from start to finish when you select "Restart Kernel and Run All Cells..". Please test this before submission.
- Use the SUTD Education Cluster to solve and test the assignment. If you work on another environment, minimally test your work on the SUTD Education Cluster.

**Rubric for assessment** 

Your submission will be graded using the following criteria. 
1. Code executes: your code should execute without errors. The SUTD Education cluster should be used to ensure the same execution environment.
2. Correctness: the code should produce the correct result or the text answer should state the factual correct answer.
3. Style: your code should be written in a way that is clean and efficient. Your text answers should be relevant, concise and easy to understand.
4. Partial marks will be awarded for partially correct solutions.
5. Creativity and innovation: in this assignment you have more freedom to design your solution, compared to the first assignments. You can show of your creativity and innovative mindset. 
6. There is a maximum of 215 points for this assignment, which will be normalized to 10% of your grade.

**ChatGPT policy** 

If you use AI tools, such as ChatGPT, Claude, Gemini, etc. to solve the assignment questions, you need to be transparent about its use and mark AI-generated content as such. In particular, you should include the following in addition to your final answer:
- A copy or screenshot of the prompt you used
- The name of the AI model
- The AI generated output
- An explanation why the answer is correct or what you had to change to arrive at the correct answer

**Assignment Notes:** Please make sure to save the notebook as you go along. Submission instructions are located at the bottom of the notebook.



### Retrieval-Augmented Generation (RAG) 

In this assignment, you will be building a Retrieval-Augmented Generation (RAG) question answering system which can answer questions about SUTD.

We'll be leveraging `langchain` and the LLM API of your choice from assignment 1

Check out the docs:
- [LangChain](https://docs.langchain.com/docs/)


The SUTD website used to allow chatting with current students. Unfortunately, this feature does not exist anymore. Let's build a chatbot to fill this gap!


# Install dependencies
Use pip to install all required dependencies of this assignment in the cell below. Make sure to test this on the SUTD cluster as different environments have different software pre-installed.  

Use of AI is documented and attached in a link here as I do not want to mess up the notebook:

link.com

In [28]:
# QUESTION: Install and import all required packages
# The rest of your code should execute without any import or dependency errors.

# **--- ADD YOUR SOLUTION HERE (10 points) ---**

%pip install -q requests beautifulsoup4 pandas

# this was ran aft chunking
%pip install -q langchain langchain-community sentence-transformers faiss-cpu transformers torch accelerate

# ran before llm
%pip install -q openai python-dotenv

# ----------------


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests, bs4, pandas
print("Deps installed")

Deps installed


# Download documents
The RAG application should be able to answer questions based on ingested documents. For the SUTD chatbot, download PDF and HTML files from the SUTD website. The documents should contain information about the admission process, available courses and the university in general.


In [ ]:
# QUESTION: Download documents from the SUTD website
# You should download at least 10 documents but more documents can increase the knowledge base of your chatbot.

# **--- ADD YOUR SOLUTION HERE (20 points) ---**



In [4]:
!python -m src.fetch_html \
  --seed data/seed_urls_pages.txt \
  --out data/raw/pages \
  --meta data/raw/pages/metadata.csv \
  --delay 2

from pathlib import Path

raw_files = list(Path("data/raw/pages").glob("*.html"))

print(f"Total HTML files fetched: {len(raw_files)}\n")

[1/5] Fetching: https://www.sutd.edu.sg/asd/education/undergraduate/curriculum
[2/5] Fetching: https://www.sutd.edu.sg/istd/education/undergraduate/curriculum/
[3/5] Fetching: https://www.sutd.edu.sg/dai/education/undergraduate/curriculum/
[4/5] Fetching: https://www.sutd.edu.sg/epd/education/undergraduate/curriculum/
[5/5] Fetching: https://www.sutd.edu.sg/esd/education/undergraduate/curriculum/beyond-ay-2019/
Done. HTML saved to data/raw/pages and metadata written to data/raw/pages/metadata.csv
Total HTML files fetched: 17



In [5]:
!python -m src.extract_page_content \
  --raw data/raw/pages \
  --out data/processed/docs \
  --meta data/processed/docs/metadata.csv

from pathlib import Path

docs = list(Path("data/processed/docs").glob("*_content.txt"))

print(f"Total cleaned documents created: {len(docs)}\n")

about_design-ai.html → about_design-ai_content.txt | chars=7779
admissions_undergraduate_admission-requirements_international-qualifications.html → admissions_undergraduate_admission-requirements_international-qualifications_content.txt | chars=4800
admissions_undergraduate_admission-requirements_overview.html → admissions_undergraduate_admission-requirements_overview_content.txt | chars=2246
admissions_undergraduate_application-guide.html → admissions_undergraduate_application-guide_content.txt | chars=3957
admissions_undergraduate_local-diploma_application-timeline.html → admissions_undergraduate_local-diploma_application-timeline_content.txt | chars=3048
admissions_undergraduate_local-diploma_criteria-for-admission.html → admissions_undergraduate_local-diploma_criteria-for-admission_content.txt | chars=5183
asd_education_undergraduate_curriculum.html → asd_education_undergraduate_curriculum_content.txt | chars=11468
campus-life_housing_freshmore.html → campus-life_housing_freshmore_

# Split documents
Use LangChain to split the documents into smaller text chunks. 

In [ ]:

# QUESTION: Use langchain to split the documents into chunks 

#--- ADD YOUR SOLUTION HERE (20 points)---



In [6]:
# building registry

from pathlib import Path
import pandas as pd

# fetch metadata from raw stage
raw_meta = pd.read_csv("data/raw/pages/metadata.csv")

# map raw html filename -> source info
raw_meta["raw_file"] = raw_meta["raw_path"].apply(lambda x: Path(x).name if pd.notna(x) else "")
raw_lookup = raw_meta.set_index("raw_file")[["url", "retrieved_at"]].to_dict("index")

rows = []

base = Path("data/processed/docs")
for domain_dir in ["academics", "admissions", "housing"]:
    folder = base / domain_dir
    for fp in sorted(folder.glob("*_content.txt")):
        # reconstruct raw html filename from processed txt filename
        raw_file = fp.name.replace("_content.txt", ".html")
        src = raw_lookup.get(raw_file, {})

        rows.append({
            "doc_id": fp.stem,
            "domain": domain_dir,
            "path": str(fp),
            "source_url": src.get("url", ""),
            "retrieved_at": src.get("retrieved_at", ""),
        })

doc_registry = pd.DataFrame(rows)
doc_registry

,doc_id,domain,path,source_url,retrieved_at
0,about_design-ai_content,academics,data/processed/docs/academics/about_design-ai_...,https://www.sutd.edu.sg/about/design-ai/,2026-03-14T08:56:49.833205+00:00
1,asd_education_undergraduate_curriculum_content,academics,data/processed/docs/academics/asd_education_un...,https://www.sutd.edu.sg/asd/education/undergra...,2026-03-14T09:15:06.623893+00:00
2,dai_education_undergraduate_curriculum_content,academics,data/processed/docs/academics/dai_education_un...,https://www.sutd.edu.sg/dai/education/undergra...,2026-03-14T09:15:10.901466+00:00
3,education_undergraduate_content,academics,data/processed/docs/academics/education_underg...,https://www.sutd.edu.sg/education/undergraduate,2026-02-24T16:53:09.057783+00:00
4,education_undergraduate_freshmore-subjects_con...,academics,data/processed/docs/academics/education_underg...,https://www.sutd.edu.sg/education/undergraduat...,2026-02-24T16:53:11.186665+00:00
5,epd_education_undergraduate_curriculum_content,academics,data/processed/docs/academics/epd_education_un...,https://www.sutd.edu.sg/epd/education/undergra...,2026-03-14T09:15:13.027442+00:00
6,esd_education_undergraduate_curriculum_beyond-...,academics,data/processed/docs/academics/esd_education_un...,https://www.sutd.edu.sg/esd/education/undergra...,2026-03-14T09:15:15.365963+00:00
7,istd_education_undergraduate_curriculum_content,academics,data/processed/docs/academics/istd_education_u...,https://www.sutd.edu.sg/istd/education/undergr...,2026-03-14T09:15:08.810728+00:00
8,media-releases-listing_sutd-becomes-worlds-fir...,academics,data/processed/docs/academics/media-releases-l...,https://www.sutd.edu.sg/media-releases-listing...,2026-03-14T08:56:44.790956+00:00
9,media-releases-listing_sutd-broadens-scope-of-...,academics,data/processed/docs/academics/media-releases-l...,https://www.sutd.edu.sg/media-releases-listing...,2026-03-14T08:56:47.465744+00:00


In [10]:
docs = []

for _, row in doc_registry.iterrows():
    text = Path(row["path"]).read_text(encoding="utf-8")
    docs.append({
        "doc_id": row["doc_id"],
        "domain": row["domain"],
        "source_url": row["source_url"],
        "retrieved_at": row["retrieved_at"],
        "text": text
    })

print(f"Loaded {len(docs)} documents.")
for i, doc in enumerate(docs, 1):
    print(f"{i}. {doc['doc_id']} - {doc['domain']}")

Loaded 17 documents.
1. about_design-ai_content - academics
2. asd_education_undergraduate_curriculum_content - academics
3. dai_education_undergraduate_curriculum_content - academics
4. education_undergraduate_content - academics
5. education_undergraduate_freshmore-subjects_content - academics
6. epd_education_undergraduate_curriculum_content - academics
7. esd_education_undergraduate_curriculum_beyond-ay-2019_content - academics
8. istd_education_undergraduate_curriculum_content - academics
9. media-releases-listing_sutd-becomes-worlds-first-university-to-incorporate-design-ai-in-all-first-year-courses_content - academics
10. media-releases-listing_sutd-broadens-scope-of-flagship-design-and-ai-degree-first-university-to-integrate-social-sciences-into-technology-degree_content - academics
11. admissions_undergraduate_admission-requirements_international-qualifications_content - admissions
12. admissions_undergraduate_admission-requirements_overview_content - admissions
13. admissions

In [11]:
# deciding on how to chunk

from pathlib import Path
import pandas as pd
import re

BASE_DIR = Path("data/processed/docs")
DOMAINS = ["academics", "admissions", "housing"]

def approx_tokens(text: str) -> int:
    # rough estimate good enough for chunking decisions
    return max(1, int(len(text.split()) * 1.3))

def split_paragraphs(text: str):
    # split on blank lines first
    paras = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    return paras

rows = []

for domain in DOMAINS:
    folder = BASE_DIR / domain
    for fp in sorted(folder.glob("*.txt")):
        text = fp.read_text(encoding="utf-8", errors="ignore")
        paras = split_paragraphs(text)

        char_count = len(text)
        word_count = len(text.split())
        token_est = approx_tokens(text)

        para_char_lengths = [len(p) for p in paras] if paras else [0]
        para_word_lengths = [len(p.split()) for p in paras] if paras else [0]

        rows.append({
            "domain": domain,
            "file_name": fp.name,
            "path": str(fp),
            "chars": char_count,
            "words": word_count,
            "approx_tokens": token_est,
            "num_paragraphs": len(paras),
            "avg_para_chars": sum(para_char_lengths) / len(para_char_lengths),
            "max_para_chars": max(para_char_lengths),
            "avg_para_words": sum(para_word_lengths) / len(para_word_lengths),
            "max_para_words": max(para_word_lengths),
            "num_lines": len(text.splitlines()),
        })

doc_stats = pd.DataFrame(rows).sort_values("words", ascending=False).reset_index(drop=True)
doc_stats

,domain,file_name,path,chars,words,approx_tokens,num_paragraphs,avg_para_chars,max_para_chars,avg_para_words,max_para_words,num_lines
0,admissions,faq_no_links.txt,data/processed/docs/admissions/faq_no_links.txt,22031,3579,4652,45,487.6,1113,79.533333,190,250
1,academics,esd_education_undergraduate_curriculum_beyond-...,data/processed/docs/academics/esd_education_un...,15794,2379,3092,1,15794.0,15794,2379.000000,2379,402
2,academics,dai_education_undergraduate_curriculum_content...,data/processed/docs/academics/dai_education_un...,12541,1899,2468,1,12541.0,12541,1899.000000,1899,335
3,academics,istd_education_undergraduate_curriculum_conten...,data/processed/docs/academics/istd_education_u...,12439,1852,2407,1,12439.0,12439,1852.000000,1852,352
4,academics,asd_education_undergraduate_curriculum_content...,data/processed/docs/academics/asd_education_un...,11468,1714,2228,1,11468.0,11468,1714.000000,1714,423
5,academics,epd_education_undergraduate_curriculum_content...,data/processed/docs/academics/epd_education_un...,9337,1429,1857,1,9337.0,9337,1429.000000,1429,371
6,academics,about_design-ai_content.txt,data/processed/docs/academics/about_design-ai_...,7779,1232,1601,1,7779.0,7779,1232.000000,1232,54
7,academics,education_undergraduate_content.txt,data/processed/docs/academics/education_underg...,7518,1047,1361,1,7518.0,7518,1047.000000,1047,165
8,academics,education_undergraduate_freshmore-subjects_con...,data/processed/docs/academics/education_underg...,7292,1027,1335,1,7292.0,7292,1027.000000,1027,99
9,academics,media-releases-listing_sutd-broadens-scope-of-...,data/processed/docs/academics/media-releases-l...,7318,1025,1332,1,7318.0,7318,1025.000000,1025,25


In [12]:
summary = doc_stats[[
    "chars", "words", "approx_tokens",
    "num_paragraphs", "avg_para_chars", "max_para_chars",
    "avg_para_words", "max_para_words"
]].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])

summary

,chars,words,approx_tokens,num_paragraphs,avg_para_chars,max_para_chars,avg_para_words,max_para_words
count,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000
mean,7752.500000,1170.222222,1520.833333,3.444444,6555.422222,6590.166667,975.807407,981.944444
std,5452.935747,866.177169,1125.927657,10.370899,4396.824473,4348.257775,662.504731,654.173072
min,1330.000000,200.000000,260.000000,1.000000,487.600000,1113.000000,79.533333,190.000000
25%,3675.000000,565.750000,735.250000,1.000000,3181.250000,3181.250000,493.500000,493.500000
50%,7305.000000,1026.000000,1333.500000,1.000000,6237.500000,6237.500000,907.000000,907.000000
75%,10935.250000,1642.750000,2135.250000,1.000000,8947.500000,8947.500000,1379.750000,1379.750000
90%,13516.900000,2043.000000,2655.200000,1.000000,12469.600000,12469.600000,1866.100000,1866.100000
95%,16729.550000,2559.000000,3326.000000,7.600000,13028.950000,13028.950000,1971.000000,1971.000000
max,22031.000000,3579.000000,4652.000000,45.000000,15794.000000,15794.000000,2379.000000,2379.000000


In [13]:
print("Top 10 longest docs by words:")
display(doc_stats[["domain", "file_name", "words", "approx_tokens", "num_paragraphs", "max_para_words"]].head(10))

print("Top 10 shortest docs by words:")
display(doc_stats[["domain", "file_name", "words", "approx_tokens", "num_paragraphs", "max_para_words"]].tail(10))

Top 10 longest docs by words:


,domain,file_name,words,approx_tokens,num_paragraphs,max_para_words
0,admissions,faq_no_links.txt,3579,4652,45,190
1,academics,esd_education_undergraduate_curriculum_beyond-...,2379,3092,1,2379
2,academics,dai_education_undergraduate_curriculum_content...,1899,2468,1,1899
3,academics,istd_education_undergraduate_curriculum_conten...,1852,2407,1,1852
4,academics,asd_education_undergraduate_curriculum_content...,1714,2228,1,1714
5,academics,epd_education_undergraduate_curriculum_content...,1429,1857,1,1429
6,academics,about_design-ai_content.txt,1232,1601,1,1232
7,academics,education_undergraduate_content.txt,1047,1361,1,1047
8,academics,education_undergraduate_freshmore-subjects_con...,1027,1335,1,1027
9,academics,media-releases-listing_sutd-broadens-scope-of-...,1025,1332,1,1025


Top 10 shortest docs by words:


,domain,file_name,words,approx_tokens,num_paragraphs,max_para_words
8,academics,education_undergraduate_freshmore-subjects_con...,1027,1335,1,1027
9,academics,media-releases-listing_sutd-broadens-scope-of-...,1025,1332,1,1025
10,admissions,admissions_undergraduate_local-diploma_criteri...,789,1025,1,789
11,admissions,admissions_undergraduate_admission-requirement...,712,925,1,712
12,admissions,admissions_undergraduate_application-guide_con...,571,742,1,571
13,academics,media-releases-listing_sutd-becomes-worlds-fir...,564,733,1,564
14,admissions,admissions_undergraduate_local-diploma_applica...,470,611,1,470
15,admissions,admissions_undergraduate_admission-requirement...,289,375,1,289
16,housing,campus-life_housing_freshmore_rooms-and-amenit...,286,371,1,286
17,housing,campus-life_housing_freshmore_content.txt,200,260,1,200


In [14]:
# shortest, median, longest
sample_indices = [
    0,                          # longest
    len(doc_stats) // 2,        # median-ish
    len(doc_stats) - 1          # shortest
]

for idx in sample_indices:
    row = doc_stats.iloc[idx]
    print("=" * 100)
    print(f"FILE: {row['file_name']}")
    print(f"DOMAIN: {row['domain']}")
    print(f"WORDS: {row['words']}, TOKENS~: {row['approx_tokens']}, PARAGRAPHS: {row['num_paragraphs']}")
    print("-" * 100)

    text = Path(row["path"]).read_text(encoding="utf-8", errors="ignore")
    print(text[:2000])
    print("\n")

FILE: faq_no_links.txt
DOMAIN: admissions
WORDS: 3579, TOKENS~: 4652, PARAGRAPHS: 45
----------------------------------------------------------------------------------------------------
--------------
If my tuition fees are fully covered by a scholarship, can I apply for living allowance only under the Study Loan (SL)?

Yes, you may apply for a living allowance under the SL if your scholarship does not provide any allowance.
--------------
Do Tuition Fee Loan (TFL) and Study Loan (SL) cover Compulsory Miscellaneous Fees (CMF) and hostel fees?

TFL and SL do not cover CMF and hostel fees. However, students may consider applying for a living allowance under the SL which can provide up to S$3,600 per year.
--------------
Do Tuition Fee Loan (TFL) and Study Loan (SL) cover full tuition fees for Singapore Permanent Residents (SPR) and International Students (IS)?

No. TFL and SL will not cover the tuition fees of SPR and IS in full as the loan amount is capped at the subsidised tuition fees

In [15]:
all_para_word_lengths = []

for _, row in doc_stats.iterrows():
    text = Path(row["path"]).read_text(encoding="utf-8", errors="ignore")
    paras = split_paragraphs(text)
    all_para_word_lengths.extend([len(p.split()) for p in paras if p.strip()])

para_df = pd.DataFrame({"para_words": all_para_word_lengths})
para_df.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])

,para_words
count,62.000000
mean,339.741935
std,540.240594
min,23.000000
25%,52.250000
50%,94.000000
75%,264.500000
90%,1045.000000
95%,1699.750000
max,2379.000000


In [16]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("data/processed/docs")
DOMAINS = ["academics", "admissions", "housing"]

docs = []

for domain in DOMAINS:
    folder = BASE_DIR / domain
    for fp in folder.glob("*.txt"):
        text = fp.read_text(encoding="utf-8", errors="ignore")

        docs.append({
            "doc_id": fp.stem,
            "domain": domain,
            "path": str(fp),
            "text": text
        })

docs_df = pd.DataFrame(docs)

print("Documents loaded:", len(docs_df))
docs_df.head()

Documents loaded: 18


,doc_id,domain,path,text
0,about_design-ai_content,academics,data/processed/docs/academics/about_design-ai_...,About\nDesign·AI\nAbout\nDesign·AI – Innovatio...
1,education_undergraduate_content,academics,data/processed/docs/academics/education_underg...,Design·AI Education\nUndergraduate studies at ...
2,istd_education_undergraduate_curriculum_content,academics,data/processed/docs/academics/istd_education_u...,Education\nUndergraduate programme\nISTD curri...
3,media-releases-listing_sutd-becomes-worlds-fir...,academics,data/processed/docs/academics/media-releases-l...,SUTD Becomes World’s First University to Incor...
4,education_undergraduate_freshmore-subjects_con...,academics,data/processed/docs/academics/education_underg...,Design·AI Education\nUndergraduate studies at ...


In [17]:
def chunk_text(text, chunk_size=900, overlap=150):
    chunks = []
    start = 0
    text = text.strip()

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        if end >= len(text):
            break

        start += chunk_size - overlap

    return chunks

In [18]:
chunk_rows = []

for _, row in docs_df.iterrows():

    chunks = chunk_text(row["text"])

    for i, chunk in enumerate(chunks):

        chunk_rows.append({
            "chunk_id": f'{row["doc_id"]}_chunk_{i:03d}',
            "doc_id": row["doc_id"],
            "domain": row["domain"],
            "text": chunk
        })

chunk_df = pd.DataFrame(chunk_rows)

print("Total documents:", len(docs_df))
print("Total chunks:", len(chunk_df))

chunk_df.head()

Total documents: 18
Total chunks: 192


,chunk_id,doc_id,domain,text
0,about_design-ai_content_chunk_000,about_design-ai_content,academics,About\nDesign·AI\nAbout\nDesign·AI – Innovatio...
1,about_design-ai_content_chunk_001,about_design-ai_content,academics,"y pedagogy, our\npivot to artificial intellige..."
2,about_design-ai_content_chunk_002,about_design-ai_content,academics,is grounded in knowing how to work with many p...
3,about_design-ai_content_chunk_003,about_design-ai_content,academics,when not to use AI is another critical part o...
4,about_design-ai_content_chunk_004,about_design-ai_content,academics,because AI is now a fellow team-mate. You int...


In [20]:
for i in range(3):
    print("-----")
    print(chunk_df.iloc[i]["chunk_id"])
    print(chunk_df.iloc[i]["text"][:500])

-----
about_design-ai_content_chunk_000
About
Design·AI
About
Design·AI – Innovation beyond human imagination
SUTD is the world’s first Design·AI university.
The AI future that used to be called Science Fiction, is Here. Is Now.
At SUTD, we are reimagining education and research for this new era. We are no longer solutioning for the “old world” with “old tools”, we are redefining education, research and innovation with Design·AI – for new world problems. As the world’s first Design·AI University, we unite design, artificial intelligen
-----
about_design-ai_content_chunk_001
y pedagogy, our
pivot to artificial intelligence (AI)
centres on the notion that AI is not just a tool, but a partner and that AI, just like technology and design, is an integral part of SUTD’s education and research.
By going beyond its traditional role as a technological tool, AI in SUTD is viewed as an invaluable part of the team, working hand-in-hand with its human counterparts, leveraging on each other’s streng

REDO CHUNKING COS WORDS CUT OFF

In [21]:
def chunk_text(text, chunk_size=900, overlap=150):
    chunks = []
    start = 0
    text = text.strip()
    length = len(text)

    while start < length:

        end = start + chunk_size
        if end >= length:
            chunks.append(text[start:].strip())
            break

        # move end forward until next whitespace
        while end < length and not text[end].isspace():
            end += 1

        chunk = text[start:end].strip()
        chunks.append(chunk)

        start = end - overlap
        if start < 0:
            start = 0

    return chunks

In [22]:
chunk_rows = []

for _, row in docs_df.iterrows():

    chunks = chunk_text(row["text"])

    for i, chunk in enumerate(chunks):

        chunk_rows.append({
            "chunk_id": f'{row["doc_id"]}_chunk_{i:03d}',
            "doc_id": row["doc_id"],
            "domain": row["domain"],
            "text": chunk
        })

chunk_df = pd.DataFrame(chunk_rows)

print("Total documents:", len(docs_df))
print("Total chunks:", len(chunk_df))

chunk_df.head()

Total documents: 18
Total chunks: 191


,chunk_id,doc_id,domain,text
0,about_design-ai_content_chunk_000,about_design-ai_content,academics,About\nDesign·AI\nAbout\nDesign·AI – Innovatio...
1,about_design-ai_content_chunk_001,about_design-ai_content,academics,"pedagogy, our\npivot to artificial intelligenc..."
2,about_design-ai_content_chunk_002,about_design-ai_content,academics,grounded in knowing how to work with many peop...
3,about_design-ai_content_chunk_003,about_design-ai_content,academics,n not to use AI is another critical part of De...
4,about_design-ai_content_chunk_004,about_design-ai_content,academics,use AI is now a fellow team-mate. You interact...


In [23]:
for i in range(3):
    print("-----")
    print(chunk_df.iloc[i]["chunk_id"])
    print(chunk_df.iloc[i]["text"][:500])

-----
about_design-ai_content_chunk_000
About
Design·AI
About
Design·AI – Innovation beyond human imagination
SUTD is the world’s first Design·AI university.
The AI future that used to be called Science Fiction, is Here. Is Now.
At SUTD, we are reimagining education and research for this new era. We are no longer solutioning for the “old world” with “old tools”, we are redefining education, research and innovation with Design·AI – for new world problems. As the world’s first Design·AI University, we unite design, artificial intelligen
-----
about_design-ai_content_chunk_001
pedagogy, our
pivot to artificial intelligence (AI)
centres on the notion that AI is not just a tool, but a partner and that AI, just like technology and design, is an integral part of SUTD’s education and research.
By going beyond its traditional role as a technological tool, AI in SUTD is viewed as an invaluable part of the team, working hand-in-hand with its human counterparts, leveraging on each other’s strength

In [25]:
# QUESTION: create embeddings of document chunks and store them in a local vector store for fast lookup
# Decide an appropriate embedding model. Use Huggingface to run the embedding model locally.
# You do not have to use cloud-based APIs.

from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Convert chunk_df into LangChain Documents
langchain_docs = []

for _, row in chunk_df.iterrows():
    langchain_docs.append(
        Document(
            page_content=row["text"],
            metadata={
                "chunk_id": row["chunk_id"],
                "doc_id": row["doc_id"],
                "domain": row["domain"],
            }
        )
    )

print(f"Total LangChain documents: {len(langchain_docs)}")

# Local HuggingFace embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Store embeddings in FAISS
vectorstore = FAISS.from_documents(langchain_docs, embedding_model)

print("FAISS vector store created successfully.")
print(f"Total chunks stored: {len(langchain_docs)}")

Total LangChain documents: 191


/var/folders/m3/5fl7_p_d3tn3nyv1qfkwf91c0000gn/T/ipykernel_35546/2679438268.py:27: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/Users/thamhaowei/Documents/GitHub/sutd_5055mlops/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 18104.66it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
em

FAISS vector store created successfully.
Total chunks stored: 191


In [26]:
vectorstore.save_local("data/vectorstore/faiss_index")
print("Vector store saved to data/vectorstore/faiss_index")

for i in range(5):
    print("=" * 100)
    print(langchain_docs[i].metadata)
    print(langchain_docs[i].page_content[:400])

Vector store saved to data/vectorstore/faiss_index
{'chunk_id': 'about_design-ai_content_chunk_000', 'doc_id': 'about_design-ai_content', 'domain': 'academics'}
About
Design·AI
About
Design·AI – Innovation beyond human imagination
SUTD is the world’s first Design·AI university.
The AI future that used to be called Science Fiction, is Here. Is Now.
At SUTD, we are reimagining education and research for this new era. We are no longer solutioning for the “old world” with “old tools”, we are redefining education, research and innovation with Design·AI – for n
{'chunk_id': 'about_design-ai_content_chunk_001', 'doc_id': 'about_design-ai_content', 'domain': 'academics'}
pedagogy, our
pivot to artificial intelligence (AI)
centres on the notion that AI is not just a tool, but a partner and that AI, just like technology and design, is an integral part of SUTD’s education and research.
By going beyond its traditional role as a technological tool, AI in SUTD is viewed as an invaluable part of the 

I used the HuggingFace embedding model sentence-transformers/all-MiniLM-L6-v2 together with the FAISS vector store. I chose this embedding model because it is lightweight, fast, runs locally, and is widely used for semantic retrieval tasks. This makes it suitable for an assignment setting where reproducibility and efficiency matter. I used FAISS because it is a simple and efficient local vector store for nearest-neighbour similarity search. It integrates well with LangChain and does not require any external service. Since the document collection is relatively small, FAISS is more than sufficient for fast lookup. This setup keeps the RAG pipeline fully local and easy to run on a laptop or cluster.

### QUESTION: 

What chunking method or strategy did you use? Why did you use this method. Explain your design decision in less than 10 sentences.


**--- ADD YOUR SOLUTION HERE (10 points) ---**


------------------------------


In [ ]:
# QUESTION: create embeddings of document chunks and store them in a local vector store for fast lookup
# Decide an appropriate embedding model. Use Huggingface to run the embedding model locally.
# You do not have to use cloud-based APIs.

#--- ADD YOUR SOLUTION HERE (20 points)---


#------------------------------


### QUESTION: 

What embeddings and vector store did you use and why? Explain your design decision in less than 10 sentences.


**--- ADD YOUR SOLUTION HERE (10 points) ---**


------------------------------



In [ ]:
# Execute a query against the vector store

query = "When was SUTD founded?"

# QUESTION: run the query against the vector store, print the top 5 search results

#--- ADD YOUR SOLUTION HERE (5 points)---

#------------------------------

In [27]:
# Execute a query against the vector store

query = "When was SUTD founded?"

# QUESTION: run the query against the vector store, print the top 5 search results

results = vectorstore.similarity_search(query, k=5)

print(f"Top {len(results)} results for query: {query}\n")

for i, doc in enumerate(results, start=1):
    print("=" * 120)
    print(f"Result {i}")
    print("Metadata:", doc.metadata)
    print("Content preview:")
    print(doc.page_content[:800])
    print()

Top 5 results for query: When was SUTD founded?

Result 1
Metadata: {'chunk_id': 'istd_education_undergraduate_curriculum_content_chunk_016', 'doc_id': 'istd_education_undergraduate_curriculum_content', 'domain': 'academics'}
Content preview:
aduates to adapt to emerging technologies and changing market demands, ensuring long-term career viability across various sectors.
Can’t find what you need?
View all FAQs
Academic calendar
Get more details about your school terms, vacation periods and more.
See more
SUTD Courses
Browse courses offered throughout SUTD, and discover the best fit for your interests.
See more

Result 2
Metadata: {'chunk_id': 'admissions_undergraduate_local-diploma_application-timeline_content_chunk_003', 'doc_id': 'admissions_undergraduate_local-diploma_application-timeline_content', 'domain': 'admissions'}
Content preview:
d modules will also be taken into consideration. You should also possess a good pass in your Additional Mathematics and Science subjects, i.e. Phy

In [ ]:
# Execute the below query against the model and let it it answer from it's internal memory

query = "What courses are available in SUTD?"


#--- ADD YOUR SOLUTION HERE (40 points)---



#------------------------------


In [29]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(Path(".env"))
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found in .env"

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def openai_text(prompt: str, model: str = "gpt-4o-mini", max_output_tokens: int = 300, temperature: float = 0.2) -> str:
    resp = client.responses.create(
        model=model,
        input=prompt,
        max_output_tokens=max_output_tokens,
        temperature=temperature,
    )
    return resp.output_text

AssertionError: OPENAI_API_KEY not found in .env

In [ ]:
# QUESTION: Now put everything together. Use langchain to integrate your vector store and Llama model into a RAG system
# Run the below example question against your RAG system.

# Example questions
query = "How can I increase my chances of admission into SUTD?"


#--- ADD YOUR SOLUTION HERE (40 points)---


#------------------------------


In [ ]:
# QUESTION: Below is set of test questions. Add another 10 test questions based on your user interviews and your value proposition canvas.
# Run the complete set of test questions against the RAG question answering system. 

questions = ["What are the admissions deadlines for SUTD?",
             "Is there financial aid available?",
             "What is the minimum score for the Mother Tongue Language?",
             "Do I require reference letters?",
             "Can polytechnic diploma students apply?",
             "Do I need SAT score?",
             "How many PhD students does SUTD have?",
             "How much are the tuition fees for Singaporeans?",
             "How much are the tuition fees for international students?",
             "Is there a minimum CAP?"
             ]

#--- ADD YOUR SOLUTION HERE (20 points)---

        
#---------------------------



### QUESTION: 


Manually inspect each answer, fact check whether the answer is correct (use Google or any other method) and check the retrieved documents

For each question, answer and context triple, record the following

- How accurate is the answer (1-5, 5 best)?
- How relevant is the retrieved context (1-5, 5 best)?
- How grounded is the answer in the retrieved context (instead of relying on the LLM's internal knowledge) (1-5, 5 best)?

**--- ADD YOUR SOLUTION HERE (20 points) ---**


------------------------------



You can try improve the chatbot by going back to previous steps in the notebook and change things until the submission deadline. For example, you can add more data sources, change the embedding models, change the data pre-processing, etc. 


# End

This concludes Individual assignment 2.

Please submit this notebook with your answers and the generated output cells as a **Jupyter notebook file** via github.

1. Create a private github repository **sutd_5055mlop** under your github user.
2. Add your instructors as collaborator: ddahlmeier, bearwithchris and MarkHershey
3. Save your submission as `individual_assignment_02_StudentID`.ipynb 
4. Push the submission files to your repo 
5. Submit the link to the repo via eDimensions


